In [0]:
import logging
from pyspark.sql.functions import current_timestamp, lit

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
log = logging.getLogger(__name__)

RAW_MARKET_PATH    = "abfss://raw@cryptodl.dfs.core.windows.net/CSV_Streaming_Data_Source_3/"
BRONZE_OUTPUT_PATH = "abfss://bronzelayer@cryptodl.dfs.core.windows.net/CSV_Streaming_Data_Source_3//datafiles/"
SCHEMA_PATH = "abfss://bronzelayer@cryptodl.dfs.core.windows.net/CSV_Streaming_Data_Source_3/_schema/"
CHECKPOINT_PATH = "abfss://bronzelayer@cryptodl.dfs.core.windows.net/CSV_Streaming_Data_Source_3//_checkpoint/"


log.info("Starting Source 3 Bronze Load")
log.info(f"Source : {RAW_MARKET_PATH}")
log.info(f"Target : {BRONZE_OUTPUT_PATH}")

market_stream = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "csv")
         .option("header", "true")
         .option("cloudFiles.schemaLocation", SCHEMA_PATH)
         .load(RAW_MARKET_PATH)
)

bronze_market = (
    market_stream
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("data_source",         lit("market_metrics_stream"))
)

query = (
    bronze_market.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .start(BRONZE_OUTPUT_PATH)
)

query.awaitTermination()
log.info(f"Stream complete — rows processed: {query.lastProgress.get('numInputRows', 'N/A')}")



In [0]:

log.info("Reading Bronze Delta for verification...")
bronze_df = spark.read.format("delta").load(BRONZE_OUTPUT_PATH)
 
total_rows       = bronze_df.count()
distinct_metrics = bronze_df.select("metric_id").distinct().count()
duplicates       = total_rows - distinct_metrics
 
log.info(f"Total rows         : {total_rows:,}")
log.info(f"Distinct metric IDs: {distinct_metrics:,}")
 
if duplicates == 0:
    log.info("No duplicates found ")
else:
    log.warning(f"Duplicates found: {duplicates}")
 
display(bronze_df.limit(20))